# 基于badcase分析的微调优化  

## 版本现状  

基于代码trainv4.py的PPO模型结果效果优异，具体回测结果如下  

测试局数              : 100000  
平均奖励              : 286.41  
奖励标准差            : 21.68  
最高奖励              : 333.13  
最低奖励              : -10.82  
平均步数              : 164.25  
成功次数              : 99486  
坠毁次数              : 194  
超时次数              : 320  
成功率                : 99.49%  
坠毁率                : 0.19%  
超时率                : 0.32%  

但是依然存在少量（0.51%）样本存在失败，本次优化基于对此类小样本的badcase特定分析，对模型进行针对优化  

## badcase分析  

对回测脚本进行修改后，得到badcase2.py，在随机100000个种子内寻找失败案例，并保存失败种子的相关信息。  

收集到的样本大致有500条badseed，并且能稳定在这些badseed上复现坏结果。  

目前发现了两类badcase：  
- 落地横移速度过大，导致底面触地的坠毁：steps80~150  
- 落地后倾斜无法判断停止，陷入循环：steps1000  

以下是两类badcase的典型badseed复现结果：

![cat](./lib/badcase2/seed_20260427_trajectory.png)  

![cat](./lib/badcase2/seed_20260914_trajectory.png)

## 优化思路  

### 采样优化

在PPO训练中，rollout的环境初始seed采样默认是均匀采样  

$$
s_0 = U(S)
$$  

将其改为混合分布采样  

$$
s_0 ~ (1-α)U(S) + αP_{(bad)}  \\
其中  \\
P_{(bad)} : badseed集合分布  \\
α : badseed采样率  \\
$$

其预期作用能增加badseed梯度权重，提升学习困难轨迹的能力  

### 线性增加  

badcase属于低频高难度样本，其瓶颈主要卡在末端收敛